# System Rekomendacyjny - SVD++ i Hybryda (Gatunki)


In [66]:
import test_users
import numpy as np
import csv
from tqdm import tqdm
import os, pickle, json
import random
from surprise import SVD, SVDpp, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split
import pandas as pd
from scipy.sparse import csr_matrix
from collections import defaultdict

test_users = test_users.test_users

### Filmy i użytkownicy

In [67]:
class Movie:
    index = {}
    name_index = {}
    inner_index = {}
    reverse_inner_index = {}
    inner_index_gen = 0
    def __init__(self, id, name):
        self.id = id
        self.name = name
        self.ratings = []
        self.genres = []
        Movie.index[id] = self
        Movie.name_index[name] = self
    def add_rating(self, rating):
        self.ratings.append(rating)
    
    
class User:
    index = {}
    def __init__(self, id):
        self.id = id
        self.ratings = {}
        User.index[id] = self
    def add_rating(self, movie, rating):
        movie.add_rating(rating)
        self.ratings[movie.id] = rating
    def __str__(self):
        str_bldr = f'{self.id}'
        return str_bldr

### Wczytanie danych

In [68]:
limit_ocen = 10000000 # Max 20,000,263. Dla testów 100k-500k.


with open('data/movie.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    next(csv_reader)
    for line in csv_reader:
        m = Movie(int(line[0]), line[1])
        if len(line) > 2:
            genres_list = line[2].split('|')
            m.genres = genres_list

with open('data/rating.csv', encoding='utf-8') as file:
    csv_reader = csv.reader(file)
    csv_reader.__next__()
    for i, line in enumerate(tqdm(csv_reader, total=limit_ocen)):
        if i >= limit_ocen:
            break 
        if not int(line[0]) in User.index.keys():
            User(int(line[0]))
        User.index[int(line[0])].add_rating(Movie.index[int(line[1])],float(line[2]))

100%|██████████| 10000000/10000000 [01:06<00:00, 151428.59it/s]


### Klasa bazowa RatingSystem

In [69]:
class RatingSystem:
    def __init__(self):
        self.users = {id : User.index[id] for id in User.index if id not in test_users}
        self.movie_ratings = {}
        for user in tqdm(self.users):
            for movie in self.users[user].ratings:
                if movie not in self.movie_ratings.keys():
                    self.movie_ratings[movie] = [self.users[user].ratings[movie]]
                else:
                    self.movie_ratings[movie].append(self.users[user].ratings[movie])
        
    def rate(self, user, movie):
        return


## SVD++

In [70]:
class SVD_156145_155941(RatingSystem):
    def __init__(self, n_factors=50, n_epochs=20, lr_all=0.007, reg_all=0.02, load_path=None, round_output=False):
        super().__init__()
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr_all = lr_all
        self.reg_all = reg_all
        self.round_output = round_output 

        self._movie_avg = {}
        self._user_avg = {}
        self._global_avg = 3.5
        self.testset = None
        self.model = None

        if load_path:
            self.load_model(load_path)
        else:
            self._prepare_and_train()

    def _prepare_and_train(self):
        print("[SVD++ 156145_155941] Przygotowanie i trening...")
        rows = []
        for u_id, user_obj in self.users.items():
            for m_id, rating in user_obj.ratings.items():
                rows.append((int(u_id), int(m_id), float(rating)))

        df = pd.DataFrame(rows, columns=["userID", "movieID", "rating"])
        reader = Reader(rating_scale=(0.5, 5.0))
        data = Dataset.load_from_df(df[["userID", "movieID", "rating"]], reader)

        trainset, testset = train_test_split(data, test_size=0.1, random_state=42)
        self.testset = testset

        all_ratings = [r for (_, _, r) in trainset.all_ratings()]
        self._global_avg = np.mean(all_ratings) if all_ratings else 3.5

        m_sums = defaultdict(list)
        u_sums = defaultdict(list)
        for (u, m, r) in trainset.all_ratings():
            m_sums[int(trainset.to_raw_iid(m))].append(r)
            u_sums[int(trainset.to_raw_uid(u))].append(r)

        self._movie_avg = {m: np.mean(v) for m, v in m_sums.items()}
        self._user_avg = {u: np.mean(v) for u, v in u_sums.items()}

        self.model = SVDpp(
            n_factors=self.n_factors,
            n_epochs=self.n_epochs,
            lr_all=self.lr_all,
            reg_all=self.reg_all,
            random_state=42,
            verbose=True
        )
        self.model.fit(trainset)

        preds = self.model.test(testset)
        rmse = accuracy.rmse(preds, verbose=False)
        print(f"[SVD++ 156145_155941] Gotowe! Test RMSE: {rmse:.4f}")

    def save_model(self, folder_path):
        os.makedirs(folder_path, exist_ok=True)
        with open(os.path.join(folder_path, "model.pkl"), "wb") as f:
            pickle.dump(self.model, f)

        meta = {
            "global_avg": self._global_avg,
            "movie_avg": {str(k): v for k, v in self._movie_avg.items()},
            "user_avg": {str(k): v for k, v in self._user_avg.items()},
            "n_factors": self.n_factors,
            "n_epochs": self.n_epochs,
            "lr_all": self.lr_all,
            "reg_all": self.reg_all
        }

        with open(os.path.join(folder_path, "meta.json"), "w") as f:
            json.dump(meta, f)

        print(f"Model SVD++ zapisany w {folder_path}")

    def load_model(self, load_path):
        with open(os.path.join(load_path, "model.pkl"), "rb") as f:
            self.model = pickle.load(f)

        with open(os.path.join(load_path, "meta.json"), "r") as f:
            meta = json.load(f)

        self._global_avg = meta["global_avg"]
        self._movie_avg = {int(k): v for k, v in meta["movie_avg"].items()}
        self._user_avg = {int(k): v for k, v in meta["user_avg"].items()}
        self.n_factors = meta.get("n_factors", self.n_factors)
        self.n_epochs = meta.get("n_epochs", self.n_epochs)
        self.lr_all = meta.get("lr_all", self.lr_all)
        self.reg_all = meta.get("reg_all", self.reg_all)

        print(f"Model SVD++ wczytany z {load_path}")

    def _round_to_half(self, x):
        return min(5.0, max(0.5, round(x * 2) / 2))

    def rate(self, user, movie):
        try:
            u_id = int(user.id) if hasattr(user, "id") else int(user)
            m_id = int(movie)

            pred = self.model.predict(u_id, m_id)

            if pred.details.get("was_impossible", False):
                raw_score = self._fallback(u_id, m_id)
            else:
                raw_score = float(pred.est)

        except Exception:
            try:
                raw_score = self._fallback(u_id, m_id)
            except Exception:
                raw_score = self._global_avg

        if self.round_output:
            return self._round_to_half(raw_score)
        else:
            return raw_score

    def _fallback(self, u_id, m_id):
        if m_id in self._movie_avg:
            return self._movie_avg[m_id]
        if u_id in self._user_avg:
            return self._user_avg[u_id]
        return self._global_avg

    def __str__(self):
        rounding_info = "Rounded" if self.round_output else "Raw"
        return f"SVD++ 155941_156145 ({rounding_info})"

### SVD++ Genre

In [71]:
class HybridGenreSVDpp(RatingSystem):
    def __init__(self, base_svd_system, weight_svd=0.85, min_genre_support=1):
        super().__init__()
        self.base_svd = base_svd_system
        self.weight_svd = weight_svd
        self.min_genre_support = min_genre_support

        self._all_genres = self._collect_all_genres()

    def _collect_all_genres(self):
        genres = set()
        for movie in Movie.index.values():
            if hasattr(movie, "genres") and movie.genres:
                genres.update(movie.genres)
        return sorted(genres)

    def _get_user_mean(self, user):
        if not user.ratings:
            return self.base_svd._global_avg
        return np.mean(list(user.ratings.values()))

    def _get_user_genre_preferences(self, user):
        if not user.ratings:
            return {}, self.base_svd._global_avg

        user_mean = self._get_user_mean(user)

        genre_deltas = defaultdict(list)

        for movie_id, rating in user.ratings.items():
            movie = Movie.index.get(movie_id)
            if movie is None or not getattr(movie, "genres", None):
                continue

            delta = rating - user_mean
            for genre in movie.genres:
                genre_deltas[genre].append(delta)

        genre_pref = {}
        for genre, vals in genre_deltas.items():
            if len(vals) >= self.min_genre_support:
                genre_pref[genre] = float(np.mean(vals))

        return genre_pref, user_mean

    def _predict_genre_component(self, user, movie_id):
        movie = Movie.index.get(movie_id)
        if movie is None or not getattr(movie, "genres", None):
            return None

        genre_pref, user_mean = self._get_user_genre_preferences(user)
        if not genre_pref:
            return None

        available = [genre_pref[g] for g in movie.genres if g in genre_pref]
        if not available:
            return None

        genre_adjustment = float(np.mean(available))
        genre_score = user_mean + genre_adjustment

        return float(np.clip(genre_score, 0.5, 5.0))

    def _round_to_half(self, x):
        return min(5.0, max(0.5, round(x * 2) / 2))

    def rate(self, user, movie):
        svd_score = self.base_svd.rate(user, movie)
        movie_id = int(movie) if not isinstance(movie, int) else movie

        genre_score = self._predict_genre_component(user, movie_id)

        if genre_score is None:
            return svd_score

        final_score = (
            self.weight_svd * svd_score
            + (1 - self.weight_svd) * genre_score
        )

        return self._round_to_half(float(np.clip(final_score, 0.5, 5.0)))

    def __str__(self):
        return f"HybridGenreSVDpp(w_svd={self.weight_svd})"

## Hybryda (Genre + SVD++)

In [72]:
class HybridGenreSVDpp(RatingSystem):
    def __init__(self, base_svd_system, weight_svd=0.85, min_genre_support=1, round_output=False):
        super().__init__()
        self.base_svd = base_svd_system
        self.weight_svd = weight_svd
        self.min_genre_support = min_genre_support
        self.round_output = round_output

        self._all_genres = self._collect_all_genres()

    def _collect_all_genres(self):
        genres = set()
        for movie in Movie.index.values():
            if hasattr(movie, "genres") and movie.genres:
                genres.update(movie.genres)
        return sorted(genres)

    def _get_user_mean(self, user):
        if not user.ratings:
            return self.base_svd._global_avg
        return np.mean(list(user.ratings.values()))

    def _get_user_genre_preferences(self, user):
        if not user.ratings:
            return {}, self.base_svd._global_avg

        user_mean = self._get_user_mean(user)

        genre_deltas = defaultdict(list)

        for movie_id, rating in user.ratings.items():
            movie = Movie.index.get(movie_id)
            if movie is None or not getattr(movie, "genres", None):
                continue

            delta = rating - user_mean
            for genre in movie.genres:
                genre_deltas[genre].append(delta)

        genre_pref = {}
        for genre, vals in genre_deltas.items():
            if len(vals) >= self.min_genre_support:
                genre_pref[genre] = float(np.mean(vals))

        return genre_pref, user_mean

    def _predict_genre_component(self, user, movie_id):
        movie = Movie.index.get(movie_id)
        if movie is None or not getattr(movie, "genres", None):
            return None

        genre_pref, user_mean = self._get_user_genre_preferences(user)
        if not genre_pref:
            return None

        available = [genre_pref[g] for g in movie.genres if g in genre_pref]
        if not available:
            return None

        genre_adjustment = float(np.mean(available))
        genre_score = user_mean + genre_adjustment

        return float(np.clip(genre_score, 0.5, 5.0))

    def _round_to_half(self, x):
        return min(5.0, max(0.5, round(x * 2) / 2))

    def rate(self, user, movie):
        svd_score = self.base_svd.rate(user, movie)
        movie_id = int(movie) if not isinstance(movie, int) else movie

        genre_score = self._predict_genre_component(user, movie_id)

        if genre_score is None:
            final_score = svd_score
        else:
            final_score = (
                self.weight_svd * svd_score
                + (1 - self.weight_svd) * genre_score
            )

        final_score = float(np.clip(final_score, 0.5, 5.0))

        if self.round_output:
            return self._round_to_half(final_score)
        else:
            return final_score

    def __str__(self):
        rounding_info = "Rounded" if self.round_output else "Raw"
        return f"HybridGenreSVDpp(w_svd={self.weight_svd}, {rounding_info})"

## SVD 

In [73]:
class SVD_Classic_156145_155941(RatingSystem):
    def __init__(self, n_factors=50, n_epochs=20, lr_all=0.007, reg_all=0.02, load_path=None, round_output=False):
        super().__init__()
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.lr_all = lr_all
        self.reg_all = reg_all
        self.round_output = round_output

        self._movie_avg = {}
        self._user_avg = {}
        self._global_avg = 3.5
        self.testset = None
        self.model = None

        if load_path:
            self.load_model(load_path)
        else:
            self._prepare_and_train()

    def _prepare_and_train(self):
        print("[SVD Classic 156145_155941] Przygotowanie i trening...")
        rows = []
        for u_id, user_obj in self.users.items():
            for m_id, rating in user_obj.ratings.items():
                rows.append((int(u_id), int(m_id), float(rating)))

        df = pd.DataFrame(rows, columns=["userID", "movieID", "rating"])
        reader = Reader(rating_scale=(0.5, 5.0))
        data = Dataset.load_from_df(df[["userID", "movieID", "rating"]], reader)

        trainset, testset = train_test_split(data, test_size=0.1, random_state=42)
        self.testset = testset

        all_ratings = [r for (_, _, r) in trainset.all_ratings()]
        self._global_avg = np.mean(all_ratings) if all_ratings else 3.5

        m_sums = defaultdict(list)
        u_sums = defaultdict(list)
        for (u, m, r) in trainset.all_ratings():
            m_sums[int(trainset.to_raw_iid(m))].append(r)
            u_sums[int(trainset.to_raw_uid(u))].append(r)

        self._movie_avg = {m: np.mean(v) for m, v in m_sums.items()}
        self._user_avg = {u: np.mean(v) for u, v in u_sums.items()}

        self.model = SVD(
            n_factors=self.n_factors,
            n_epochs=self.n_epochs,
            lr_all=self.lr_all,
            reg_all=self.reg_all,
            random_state=42,
            verbose=True
        )
        self.model.fit(trainset)

        preds = self.model.test(testset)
        rmse = accuracy.rmse(preds, verbose=False)
        print(f"[SVD Classic 156145_155941] Gotowe! Test RMSE: {rmse:.4f}")

    def save_model(self, folder_path):
        os.makedirs(folder_path, exist_ok=True)
        with open(os.path.join(folder_path, "model.pkl"), "wb") as f:
            pickle.dump(self.model, f)

        meta = {
            "global_avg": self._global_avg,
            "movie_avg": {str(k): v for k, v in self._movie_avg.items()},
            "user_avg": {str(k): v for k, v in self._user_avg.items()},
            "n_factors": self.n_factors,
            "n_epochs": self.n_epochs,
            "lr_all": self.lr_all,
            "reg_all": self.reg_all
        }

        with open(os.path.join(folder_path, "meta.json"), "w") as f:
            json.dump(meta, f)

        print(f"Model SVD Classic zapisany w {folder_path}")

    def load_model(self, load_path):
        with open(os.path.join(load_path, "model.pkl"), "rb") as f:
            self.model = pickle.load(f)

        with open(os.path.join(load_path, "meta.json"), "r") as f:
            meta = json.load(f)

        self._global_avg = meta["global_avg"]
        self._movie_avg = {int(k): v for k, v in meta["movie_avg"].items()}
        self._user_avg = {int(k): v for k, v in meta["user_avg"].items()}
        self.n_factors = meta.get("n_factors", self.n_factors)
        self.n_epochs = meta.get("n_epochs", self.n_epochs)
        self.lr_all = meta.get("lr_all", self.lr_all)
        self.reg_all = meta.get("reg_all", self.reg_all)

        print(f"Model SVD Classic wczytany z {load_path}")

    def _round_to_half(self, x):
        return min(5.0, max(0.5, round(x * 2) / 2))

    def rate(self, user, movie):
        try:
            u_id = int(user.id) if hasattr(user, "id") else int(user)
            m_id = int(movie)

            pred = self.model.predict(u_id, m_id)

            if pred.details.get("was_impossible", False):
                raw_score = self._fallback(u_id, m_id)
            else:
                raw_score = float(pred.est)

        except Exception:
            try:
                raw_score = self._fallback(u_id, m_id)
            except Exception:
                raw_score = self._global_avg

        if self.round_output:
            return self._round_to_half(raw_score)
        else:
            return raw_score

    def _fallback(self, u_id, m_id):
        if m_id in self._movie_avg:
            return self._movie_avg[m_id]
        if u_id in self._user_avg:
            return self._user_avg[u_id]
        return self._global_avg

    def __str__(self):
        rounding_info = "Rounded" if self.round_output else "Raw"
        return f"SVD Classic ({rounding_info})"

## model gatunkowy

In [74]:
class GenreBasedRating(RatingSystem):
    def __init__(self, round_output=False):
        super().__init__()
        self.round_output = round_output
        self._global_avg = self._compute_global_avg()

    def _compute_global_avg(self):
        ratings = []
        for user in self.users.values():
            ratings.extend(user.ratings.values())
        return float(np.mean(ratings)) if ratings else 3.5

    def _round_to_half(self, x):
        return min(5.0, max(0.5, round(x * 2) / 2))

    def _get_user_avg(self, user):
        if not user.ratings:
            return self._global_avg
        return float(np.mean(list(user.ratings.values())))

    def _get_user_genre_means(self, user):
        genre_ratings = defaultdict(list)

        for movie_id, rating in user.ratings.items():
            movie = Movie.index.get(movie_id)
            if movie is None or not getattr(movie, "genres", None):
                continue

            for genre in movie.genres:
                genre_ratings[genre].append(rating)

        return {genre: float(np.mean(vals)) for genre, vals in genre_ratings.items() if vals}

    def rate(self, user, movie):
        try:
            movie_id = int(movie)
            movie_obj = Movie.index.get(movie_id)

            if movie_obj is None or not getattr(movie_obj, "genres", None):
                raw_score = self._get_user_avg(user)
            else:
                user_genre_means = self._get_user_genre_means(user)
                genre_scores = [user_genre_means[g] for g in movie_obj.genres if g in user_genre_means]

                if genre_scores:
                    raw_score = float(np.mean(genre_scores))
                else:
                    raw_score = self._get_user_avg(user)

        except Exception:
            raw_score = self._global_avg

        raw_score = float(np.clip(raw_score, 0.5, 5.0))

        if self.round_output:
            return self._round_to_half(raw_score)
        else:
            return raw_score

    def __str__(self):
        rounding_info = "Rounded" if self.round_output else "Raw"
        return f"Genre Based Rating ({rounding_info})"

### Przykłady prostych systemów oceniających


In [75]:
class NaiveRating(RatingSystem):
    """
    Przykładowy system - naiwny. 
    Hipoteza: jeżeli zwrócę każdemu filmowi średnią ocenę (2.5/5), to moja ocena będzie niezła.
    """
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        return 2.5
    def __str__(self):
        return 'Naive Rating'

class AverageMovieRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        """
        Przykładowy system - średnia.
        Hipoteza: jeżeli zwrocę każdeu filmowi średnią ocenę (wynikającą z wszystkich ocen), to moja ocena będzie niezła.
        Jeżeli ten film jeszcze nie był oceniony, to zwrócę 2.5.
        """
        n = len(self.movie_ratings[movie])
        if n == 0:
            return 2.5
        else:
            return sum(self.movie_ratings[movie])/n
    def __str__(self):
        return 'Average Movie Rating'
class AverageUserRating(RatingSystem):
    def __init__(self):
        super().__init__()
    def rate(self, user, movie):
        """
        Przykładowy system - średnia użytkownika.
        Hipoteza: jeżeli zwrócę dla tego filmu średnią ocenę wystawioną przez użytkownika, to mój system będzie niezły.

        """
        n = len(user.ratings.values())
        if n == 0:
            return 2.5
        else:
            return sum(user.ratings.values())/n
    def __str__(self):
        return 'Average User Rating'

class GlobalAverageMovieRating(RatingSystem):
    def __init__(self):
        """
        Przykładowy system - średnia ocena filmu.
        Hipoteza: średnia ocena tego filmu wśród wszystkich użytkowników powinna być dobrą estymacją.
        """
        super().__init__()
        self.GlobalAverageMovieRating = 0
        self.TotalMovies = 0
        for movie in self.movie_ratings:
            for rating  in self.movie_ratings[movie]:
                self.GlobalAverageMovieRating += rating
                self.TotalMovies += 1
        self.GlobalAverageMovieRating /= self.TotalMovies

    def rate(self, user, movie):
        return self.GlobalAverageMovieRating
    def __str__(self):
        return 'Average Global Movie Rating'
    
class Cheater(RatingSystem):
    def __init__(self):
        super().__init__()

    def rate(self, user, movie):
        """
        Testowy system.
        Jeżeli ten system działa, to coś jest nie tak - systemy mają dostęp do ocen filmów, które mają wyznaczyć - powinien działać mniej więcej tak samo jak system naiwny.
        """
        if movie in user.ratings:
            return user.ratings[movie]
        else:
            return 2.5
    def __str__(self):
        return 'Cheater'


### Ewaluacja

In [76]:
class RatingSystemEvaluator:
    def __init__(self, users=None, n_rounds=5, test_size=0.1, random_state=42, min_user_ratings=2):
        self.users = users if users is not None else {
            uid: User.index[uid] for uid in User.index if uid not in test_users
        }
        self.n_rounds = n_rounds
        self.test_size = test_size
        self.random_state = random_state
        self.min_user_ratings = min_user_ratings
        self.systems = []

        self.all_ratings = self._collect_all_ratings()

    def register(self, system):
        self.systems.append(system)

    def _collect_all_ratings(self):
        ratings = []
        for uid, user in self.users.items():
            if len(user.ratings) < self.min_user_ratings:
                continue
            for movie_id, rating in user.ratings.items():
                ratings.append((uid, movie_id, float(rating)))
        return ratings

    def _split_train_test(self, rng):
        n_total = len(self.all_ratings)
        n_test = int(n_total * self.test_size)

        indices = np.arange(n_total)
        rng.shuffle(indices)

        test_idx = set(indices[:n_test])
        test_ratings = [self.all_ratings[i] for i in test_idx]

        return test_ratings

    def _evaluate_system_on_split(self, system, test_ratings):
        sq_err = []
        abs_err = []

        for uid, movie_id, true_rating in test_ratings:
            user = self.users[uid]

            orig = user.ratings.pop(movie_id, None)

            try:
                pred = system.rate(user, movie_id)
            except Exception:
                pred = true_rating

            if orig is not None:
                user.ratings[movie_id] = orig

            sq_err.append((true_rating - pred) ** 2)
            abs_err.append(abs(true_rating - pred))

        return {
            "rmse": float(np.sqrt(np.mean(sq_err))),
            "mae": float(np.mean(abs_err))
        }

    def evaluate(self, verbose=True):
        all_results = []

        for round_idx in range(self.n_rounds):
            rng = np.random.default_rng(self.random_state + round_idx)
            test_ratings = self._split_train_test(rng)

            if verbose:
                print(f"\n=== RUNDA {round_idx + 1} / {self.n_rounds} ===")
                print(f"Liczba ocen testowych: {len(test_ratings)}")

            round_rows = []

            for system in self.systems:
                metrics = self._evaluate_system_on_split(system, test_ratings)

                row = {
                    "round": round_idx + 1,
                    "system": str(system),
                    "rmse": metrics["rmse"],
                    "mae": metrics["mae"]
                }
                all_results.append(row)
                round_rows.append(row)

            if verbose:
                df_round = pd.DataFrame(round_rows).sort_values("rmse")
                print(df_round[["system", "rmse", "mae"]].to_string(index=False))

        df_all = pd.DataFrame(all_results)

        summary = (
            df_all.groupby("system")
            .agg(
                rmse_mean=("rmse", "mean"),
                rmse_std=("rmse", "std"),
                mae_mean=("mae", "mean"),
                mae_std=("mae", "std")
            )
            .reset_index()
            .sort_values("rmse_mean")
        )

        print("\n=== PODSUMOWANIE CROSS-VALIDATION ===")
        print(summary.to_string(index=False))

        return summary

 ### Ocena systemów

 Poniższe klasy dotyczą oceny systemów - zwróć uwagę na parametr verbose, służy on do ograniczania informacji zwrotnej

In [77]:
class RatingSystemCompetition:
    
    def __init__(self):
        self.registered_systems = []
        self.users = {id : User.index[id] for id in User.index if id not in test_users}
        self.verbose = 2
    def register(self, system):
        self.registered_systems.append(system)
        
    def build_round_robin(self):
        self.pairs = {}
        for system in self.registered_systems:
            self.pairs[system]  = []
            for competitor in self.registered_systems:
                if str(system) != str(competitor):
                    self.pairs[system].append((system, competitor))

            
    def runMatch(self, system, competitor):
        users_ids = np.random.choice(np.array(list(self.users.keys())), size=100)
        score = 0
        wins = 0
        loses = 0
        draws = 0
        for user_id in users_ids:
            user = self.users[user_id]
            user_copy = copy.deepcopy(self.users[user_id])
            movie_id = np.random.choice(np.array(list(user.ratings.keys())), size=1)[0]
            del user_copy.ratings[movie_id]
            true_rating = self.users[user_id].ratings[movie_id]
            system_rating = system.rate(user_copy,movie_id)
            competitor_rating = competitor.rate(user_copy,movie_id)
            
            if abs(true_rating - system_rating) <  abs(true_rating - competitor_rating):
                score += 1
                wins += 1
            elif abs(true_rating - system_rating) >  abs(true_rating - competitor_rating):
                score -= 1
                loses += 1
            else:
                draws += 1
                
        return score, wins, draws, loses
    
    def compete(self):
        self.total_scores = {}
        for system in self.pairs:
            self.total_scores[system] = 0
            if self.verbose >= 2: print(f'{system} analysis: ')
            for matchup in self.pairs[system]:
                score, wins, draws, loses = self.runMatch(matchup[0],matchup[1])
                if self.verbose >= 2: print(f'{matchup[0]} vs {matchup[1]} : {score} ({wins} wins, {draws} draws, {loses} loses)')
                self.total_scores[system] += score
            if self.verbose >= 2: print(f'{system} score: {self.total_scores[system]}')
        if self.verbose >= 1:
            print('Final scores: ')
            place = 1
            for system in sorted(self.total_scores, key=self.total_scores.get, reverse=True):
                print(f'{place}. {system}, {self.total_scores[system]} pkt')
                place += 1

## Uruchomienie

In [ ]:
np.random.seed(42)
os.makedirs("saved_models", exist_ok=True)

svd_path = f"saved_models/svd_improved_{limit_ocen}"
svd_classic_path = f"saved_models/svd_classic_{limit_ocen}"

print("=== INICJALIZACJA MODELI NAIWNYCH ===")
global_avg = GlobalAverageMovieRating()
naive = NaiveRating()
avg_movie = AverageMovieRating()
avg_user = AverageUserRating()
cheater = Cheater()

print("\n=== INICJALIZACJA MODELI GATUNKOWYCH ===")
genre_based_raw = GenreBasedRating(round_output=False)
genre_based_rounded = GenreBasedRating(round_output=True)

print("\n--- Trenowanie SVD++ ---")
if not os.path.exists(svd_path):
    svd_pp_raw = SVD_156145_155941(n_factors=50, n_epochs=20, lr_all=0.007, reg_all=0.02, round_output=False)
    svd_pp_raw.save_model(svd_path)
else:
    svd_pp_raw = SVD_156145_155941(load_path=svd_path, round_output=False)

svd_pp_rounded = SVD_156145_155941(load_path=svd_path, round_output=True)


print("\n--- Trenowanie Klasycznego SVD ---")
if not os.path.exists(svd_classic_path):
    svd_classic_raw = SVD_Classic_156145_155941(n_factors=50, n_epochs=20, lr_all=0.007, reg_all=0.02, round_output=False)
    svd_classic_raw.save_model(svd_classic_path)
else:
    svd_classic_raw = SVD_Classic_156145_155941(load_path=svd_classic_path, round_output=False)

svd_classic_rounded = SVD_Classic_156145_155941(load_path=svd_classic_path, round_output=True)


print("\n--- Inicjalizacja Hybryd ---")
hyb_svdpp_raw = HybridGenreSVDpp(svd_pp_raw, weight_svd=0.85, min_genre_support=2, round_output=False)
hyb_svdpp_rounded = HybridGenreSVDpp(svd_pp_raw, weight_svd=0.85, min_genre_support=2, round_output=True)

try:
    hyb_rating = HybridGenreRating(svd_pp_raw, weight_svd=0.85, genre_strength=1.0)
except TypeError:
    hyb_rating = HybridGenreRating(svd_pp_raw, weight_svd=0.85)
'''
# Wszystkie systemy rekomendacyjne
systems_to_test = [
    global_avg, naive, avg_movie, avg_user, 
    genre_based_raw, genre_based_rounded, 
    svd_classic_raw, svd_classic_rounded, 
    svd_pp_raw, svd_pp_rounded, 
    hyb_svdpp_raw, hyb_svdpp_rounded, 
    hyb_rating
]
'''
# Nasze systemy rekomentacyjne
systems_to_test = [
    genre_based_raw, genre_based_rounded, 
    svd_classic_raw, svd_classic_rounded, 
    svd_pp_raw, svd_pp_rounded, 
    hyb_svdpp_raw, hyb_svdpp_rounded, 
    hyb_rating

]
print("\n" + "="*50)
print("WIELKI TURNIEJ ROUND-ROBIN (RAW vs ROUNDED)")
print("="*50)

comp = RatingSystemCompetition()
comp.verbose = 2 # 1 - tylko ostateczny ranking 2 - full info o każdej walce

for sys in systems_to_test:
    comp.register(sys)

comp.build_round_robin()
comp.compete()

=== INICJALIZACJA MODELI NAIWNYCH ===


100%|██████████| 69139/69139 [00:04<00:00, 16037.09it/s]



=== INICJALIZACJA MODELI GATUNKOWYCH ===


100%|██████████| 69139/69139 [00:04<00:00, 14334.18it/s]



--- Trenowanie SVD++ ---


100%|██████████| 69139/69139 [00:04<00:00, 15837.28it/s]


[SVD++ 156145_155941] Przygotowanie i trening...
 processing epoch 0


In [ ]:
print("\n" + "="*50)
print("KLASYCZNA EWALUACJA (RMSE & MAE)")
print("="*50)

evaluator = RatingSystemEvaluator(n_rounds=1, test_size=0.1, random_state=42)
for sys in systems_to_test:
    evaluator.register(sys)

summary_df = evaluator.evaluate(verbose=True)




KLASYCZNA EWALUACJA (RMSE & MAE)

=== RUNDA 1 / 1 ===
Liczba ocen testowych: 200000
                               system     rmse      mae
                    SVD Classic (Raw) 0.665375 0.511374
            SVD++ 155941_156145 (Raw) 0.665773 0.508838
    HybridGenreSVDpp(w_svd=0.85, Raw) 0.677940 0.520930
          HybridGenre(w=0.85, gs=1.0) 0.679313 0.521623
                SVD Classic (Rounded) 0.680379 0.496742
        SVD++ 155941_156145 (Rounded) 0.680918 0.494667
HybridGenreSVDpp(w_svd=0.85, Rounded) 0.693124 0.506463
             Genre Based Rating (Raw) 0.935456 0.726379
         Genre Based Rating (Rounded) 0.946057 0.716657

=== PODSUMOWANIE CROSS-VALIDATION ===
                               system  rmse_mean  rmse_std  mae_mean  mae_std
                    SVD Classic (Raw)   0.665375       NaN  0.511374      NaN
            SVD++ 155941_156145 (Raw)   0.665773       NaN  0.508838      NaN
    HybridGenreSVDpp(w_svd=0.85, Raw)   0.677940       NaN  0.520930      NaN
    